# Week 24, Day 3: Designing good tests

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w024d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def area(width, height):
    return width * width          # a bug: should be width * height

def test_area():
    assert area(3, 3) == 9

def test_area_again():
    assert area(5, 5) == 25

test_area(); test_area_again()
print("tests passed")

In [ ]:
from datetime import datetime

def greeting():
    hour = datetime.now().hour
    if hour < 12:
        return "Good morning"
    if hour < 18:
        return "Good afternoon"
    return "Good evening"

print(greeting())    # what should a test expect? It depends on when it runs.

In [ ]:
from datetime import datetime

def greeting(now):
    if now.hour < 12:
        return "Good morning"
    if now.hour < 18:
        return "Good afternoon"
    return "Good evening"

def test_morning():
    assert greeting(datetime(2026, 1, 1, 9, 0)) == "Good morning"

def test_six_pm_is_evening():
    assert greeting(datetime(2026, 1, 1, 18, 0)) == "Good evening"

test_morning(); test_six_pm_is_evening()
print(greeting(datetime.now()))      # the real program passes the real time

In [ ]:
import random
from collections import Counter

def my_sort(items):
    return sorted(set(items))         # a bug: loses repeated items

def test_sort_properties():
    rng = random.Random(1)            # a fixed seed keeps the test repeatable
    for _ in range(200):
        items = [rng.randint(-5, 5) for _ in range(rng.randint(0, 8))]
        result = my_sort(items)
        assert all(a <= b for a, b in zip(result, result[1:])), items
        assert Counter(result) == Counter(items), items

try:
    test_sort_properties()
except AssertionError as e:
    print("failed for input", e)

> **Key idea.** Choose inputs from each equivalence class, on and around each boundary, and from special cases. Good tests are fast, independent, repeatable, self-checking and focused, and they test behaviour, with expected values that don't copy the code's logic. Make code testable by passing in the clock, randomness and data. Property tests check a rule over many generated inputs.

### Exercise: Classes and boundaries

`parcel_price(kg)` in `parcels.py` follows the table in the lesson. Write tests (parametrize helps) that pass on the correct code and catch all six broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('parcels.py', 'w', encoding='utf-8') as f:
    f.write('def parcel_price(kg):\n    if kg <= 0 or kg > 20:\n        raise ValueError("weight must be over 0 and at most 20 kg")\n    if kg <= 1:\n        return 3.50\n    if kg <= 5:\n        return 6.00\n    return 12.00\n')

In [ ]:
import pytest
from parcels import parcel_price

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"parcels.py\").read()\nBROKEN = {\n    \"1 kg priced as medium\": GOOD.replace(\"kg <= 1:\", \"kg < 1:\"),\n    \"5 kg priced as large\": GOOD.replace(\"kg <= 5:\", \"kg < 5:\"),\n    \"20 kg rejected\": GOOD.replace(\"kg > 20\", \"kg >= 20\"),\n    \"0 kg allowed\": GOOD.replace(\"kg <= 0 or\", \"kg < 0 or\"),\n    \"no upper limit\": GOOD.replace(\" or kg > 20\", \"\"),\n    \"medium price wrong\": GOOD.replace(\"return 6.00\", \"return 6.50\"),\n}\ndef test_catches():\n    \"your tests pass on the correct code and catch all six broken versions\"\n    results = against(source, \"parcels\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"parcels.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Pass in the clock

`is_open()` uses the real time, so it can't be tested reliably. Rewrite it as `is_open(now)`, taking a `datetime`. The shop is open Monday to Saturday from 9:00 up to (not including) 17:30, and closed on Sunday. (`now.weekday()` is 0 for Monday up to 6 for Sunday.) Then add at least three tests of your own in the same editor.

In [ ]:
from datetime import datetime

def is_open():
    now = datetime.now()
    if now.weekday() == 6:
        return False
    return (9, 0) <= (now.hour, now.minute) < (17, 30)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import datetime as _dt\ndef test_signature():\n    \"is_open takes the time as a parameter\"\n    import inspect\n    assert len(inspect.signature(is_open).parameters) == 1\ndef test_behaviour():\n    \"is_open gives the right answer\"\n    cases = [(_dt(2026, 11, 2, 9, 0), True), (_dt(2026, 11, 2, 8, 59), False), (_dt(2026, 11, 7, 17, 29), True),\n             (_dt(2026, 11, 7, 17, 30), False), (_dt(2026, 11, 8, 12, 0), False), (_dt(2026, 11, 4, 13, 0), True)]\n    for when, want in cases:\n        assert is_open(when) is want, f\"is_open({when}) should be {want}\"\ndef test_yours():\n    \"you wrote at least three passing tests\"\n    r = pytest_run(source)\n    assert len(r[\"passed\"]) >= 3 and not r[\"failed\"], f\"passed: {r['passed']}, failed: {r['failed']}\"")

### Exercise: Property test a sort

Write a property test for `my_sort` in `sorting.py`: generate at least 100 random lists with a seeded `random.Random`, and check that each result is in order and has the same items as the input. It must pass on the correct code and catch all four broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('sorting.py', 'w', encoding='utf-8') as f:
    f.write('def my_sort(items):\n    result = list(items)\n    for i in range(1, len(result)):\n        j = i\n        while j > 0 and result[j - 1] > result[j]:\n            result[j - 1], result[j] = result[j], result[j - 1]\n            j -= 1\n    return result\n')

In [ ]:
import random
from collections import Counter
from sorting import my_sort

def test_sort_properties():
    rng = random.Random(1)
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"sorting.py\").read()\nBROKEN = {\n    \"drops repeats\": \"def my_sort(items):\\n    return sorted(set(items))\\n\",\n    \"backwards\": \"def my_sort(items):\\n    return sorted(items, reverse=True)\\n\",\n    \"loses the last item\": \"def my_sort(items):\\n    return sorted(items)[:-1] if len(items) > 4 else sorted(items)\\n\",\n    \"only one pass\": GOOD.replace(\"j -= 1\", \"j = 0\"),\n}\ndef test_catches():\n    \"your property test passes on the correct code and catches every broken version\"\n    assert \"Random(\" in source, \"use a seeded random.Random\"\n    results = against(source, \"sorting\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"sorting.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: A round trip

`encode(text, shift)` and `decode(text, shift)` in `cipher.py` shift letters round the alphabet (a Caesar cipher), keeping case and leaving everything else alone. Write a round-trip property test (`decode(encode(t, k), k) == t` for many random texts and shifts) **and** at least one example test with a known answer. Together they must catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('cipher.py', 'w', encoding='utf-8') as f:
    f.write('def _shift(c, k):\n    if c.isascii() and c.isalpha():\n        base = ord("a") if c.islower() else ord("A")\n        return chr((ord(c) - base + k) % 26 + base)\n    return c\n\ndef encode(text, shift):\n    return "".join(_shift(c, shift) for c in text)\n\ndef decode(text, shift):\n    return encode(text, -shift)\n')

In [ ]:
import random
from cipher import encode, decode

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"cipher.py\").read()\nBROKEN = {\n    \"no wrap round\": GOOD.replace(\"(ord(c) - base + k) % 26 + base\", \"ord(c) + k\"),\n    \"loses case\": GOOD.replace('base = ord(\"a\") if c.islower() else ord(\"A\")', 'base = ord(\"a\"); c = c.lower()'),\n    \"does nothing\": GOOD.replace('return \"\".join(_shift(c, shift) for c in text)', \"return text\"),\n    \"shifts digits too\": GOOD.replace(\"if c.isascii() and c.isalpha():\", \"if c.isascii() and c.isalnum():\"),\n}\ndef test_style():\n    \"a property test with random inputs, and an example test\"\n    assert \"Random(\" in source, \"use a seeded random.Random for the property test\"\n    r = pytest_run(source)\n    assert len(r[\"passed\"]) >= 2, \"write at least two tests\"\ndef test_catches():\n    \"your tests pass on the correct code and catch every broken version\"\n    results = against(source, \"cipher\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"cipher.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Good test or bad test?

For each test described, write the **main problem** with it, using one of: `"ok"`, `"flaky"`, `"implementation"` (tests how, not what), `"copies logic"`, `"not independent"`, `"not self-checking"`.

In [ ]:
a = ___   # prints the result of total(cart) so you can look at it
b = ___   # checks that greeting() returns "Good morning", using the real clock
c = ___   # expects price * 1.2, worked out in the test exactly as the code does it
d = ___   # checks that parse() calls the helper function _split exactly twice
e = ___   # relies on the user created by the previous test
f = ___   # checks parcel_price(5) == 6.00, a value from the price table

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (a, b, c, d, e, f) == (\"not self-checking\", \"flaky\", \"copies logic\", \"implementation\", \"not independent\", \"ok\")")

### Bug hunt: The test copies the bug

This test passes, but `add_vat` is wrong: VAT is 20%, so £80 should become £96. The test works out its expected value with the same mistake. Fix the test so its expected value is worked out independently; it should then fail on this code (that's right: leave the function alone).

In [ ]:
import pytest

def add_vat(price):
    return price * 1.02

def test_add_vat():
    price = 80
    assert add_vat(price) == pytest.approx(price * 1.02)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_independent():\n    \"the test now catches the bug, and passes on correct code\"\n    import ast\n    tests_only = \"import pytest\\nfrom vat import add_vat\\n\" + \"\\n\".join(\n        ast.unparse(n) for n in ast.parse(source).body if isinstance(n, ast.FunctionDef) and n.name.startswith(\"test_\"))\n    r = against(tests_only, \"vat\", {\"buggy\": \"def add_vat(price):\\n    return price * 1.02\\n\",\n                                    \"correct\": \"def add_vat(price):\\n    return price * 1.2\\n\"})\n    assert r[\"buggy\"], \"the test still passes on the buggy code\"\n    assert r[\"correct\"] == [], \"the test should pass on correct code (80 becomes 96)\"")

### Bug hunt: Tests that need each other

These tests pass when run in order, but `test_remove` fails when run on its own, because it depends on `test_add` having run first. Make each test set up its own data, so either one passes alone.

In [ ]:
basket = []

def test_add():
    basket.append("apple")
    assert basket == ["apple"]

def test_remove():
    basket.remove("apple")
    assert basket == []

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast as _ast\ndef _alone(name):\n    tree = _ast.parse(source)\n    keep = [n for n in tree.body if not (isinstance(n, _ast.FunctionDef) and n.name.startswith(\"test_\") and n.name != name)]\n    return pytest_run(_ast.unparse(_ast.Module(body=keep, type_ignores=[])))\ndef test_each_alone():\n    \"each test passes when run alone\"\n    for name in (\"test_add\", \"test_remove\"):\n        r = _alone(name)\n        assert r[\"passed\"] == [name], f\"{name} doesn't pass on its own\"\ndef test_together():\n    \"they pass together\"\n    assert len(pytest_run(source)[\"passed\"]) == 2")

### Puzzle: A seeded property test

Arrange the lines into a property test that dice rolls are always 1 to 6. One line is a trap.

(Put these lines in order on the website.)

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: test code that uses chance

`roll_dice(n, rng)` in `dice.py` returns a list of `n` dice rolls (each 1 to 6) made with the random generator `rng`, and raises `ValueError` if `n` is negative. Write tests, without any fixed expected list of rolls (they'd depend on how Python's generator works), that pass on the correct code and catch all **six** broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('dice.py', 'w', encoding='utf-8') as f:
    f.write('def roll_dice(n, rng):\n    if n < 0:\n        raise ValueError("n can\'t be negative")\n    return [rng.randint(1, 6) for _ in range(n)]\n')

In [ ]:
import random
import pytest
from dice import roll_dice

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"dice.py\").read()\nBROKEN = {\n    \"0 to 5\": GOOD.replace(\"rng.randint(1, 6)\", \"rng.randint(0, 5)\"),\n    \"1 to 7\": GOOD.replace(\"rng.randint(1, 6)\", \"rng.randint(1, 7)\"),\n    \"one too many\": GOOD.replace(\"range(n)\", \"range(n + 1)\"),\n    \"ignores rng\": \"import random\\n\" + GOOD.replace(\"rng.randint(1, 6)\", \"random.randint(1, 6)\"),\n    \"never rolls 6\": GOOD.replace(\"rng.randint(1, 6)\", \"rng.randint(1, 5)\"),\n    \"negative allowed\": GOOD.replace('    if n < 0:\\n        raise ValueError(\"n can\\'t be negative\")\\n', \"\"),\n}\ndef test_style():\n    \"no rolls are hard-coded\"\n    import re as _re\n    assert not _re.search(r\"==\\s*\\[\\s*\\d+\\s*,\", source), \"don't compare with a fixed list of rolls\"\ndef test_catches():\n    \"your tests pass on the correct code and catch all six broken versions\"\n    results = against(source, \"dice\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"dice.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

## Warm-up

Two tests, both passing, and the function is wrong. Both tests used squares, where width and height are equal, so the bug couldn't show. More tests of the same kind of input don't help. Good tests use **different kinds** of input.

## Choosing inputs

You can't test every input, so choose them on purpose. Two techniques do most of the work.

**Equivalence classes.** Group the inputs that the code should treat the same way, and test one from each group. For a parcel price based on weight:

| Weight | Price |
|---|---|
| up to 1 kg | £3.50 |
| over 1 kg, up to 5 kg | £6.00 |
| over 5 kg, up to 20 kg | £12.00 |
| over 20 kg, or 0 kg or less | `ValueError` |

there are five classes: invalid low, small, medium, large and invalid high. One test from each (say 0.5, 3, 10, 25 and -1) covers every rule once.

**Boundary values.** Bugs cluster at the edges between classes, because that's where `<` and `<=` get mixed up. So also test **on** each boundary, and just either side of it: 1 and 1.01, 5 and 5.01, 20 and 20.01, and 0. For whole numbers, the boundary and its neighbours (39, 40, 41).

Then add **special cases** the table doesn't mention: empty input, a single item, repeats, very large values, unusual text (spaces, accents, an empty string).

## What makes a test good

A good test is:

- **Fast.** Milliseconds. Slow tests get run less often, and then they stop protecting you.
- **Independent.** It doesn't depend on another test running first, or on the order tests run in. Each test sets up its own data.
- **Repeatable.** Same result every time, on every computer: no dependence on today's date, on chance, on the network, or on files left by an earlier run. A test that sometimes fails for no reason is called **flaky**, and people soon learn to ignore it.
- **Self-checking.** It passes or fails by itself, with `assert`. A test that prints and expects you to read is not a test.
- **Focused.** It checks one behaviour, and its name says which: `test_overdraw_raises`, `test_empty_cart_total_is_zero`. When it fails, the name is the first clue.

Two more habits matter just as much:

**Test behaviour, not implementation.** Check what a function **returns** or **does**, not how it does it internally. If you rewrite a loop as a comprehension, no test should break. Tests that check internal details break on every tidy-up, and people stop tidying.

**Don't repeat the code's logic in the test.** If the expected value is worked out with the same formula as the code, the test copies the code's mistakes:

```
def test_vat():                          # weak: the same formula, the same bug
    price = 80
    assert add_vat(price) == price * 1.2

def test_vat():                          # strong: worked out by hand
    assert add_vat(80) == pytest.approx(96)
```

Expected values should come from somewhere else: worked out by hand, from the specification, or from a simple example you're sure of. And keep tests simple, without loops and `if` statements of their own where you can, since a test with logic in it can have bugs too. (Property tests, below, are the exception that proves the rule: their loop is the point.)

## Making code testable

Some code is hard to test because it depends on things you don't control:

The fix is to **pass in** what the code depends on, rather than having it reach out and get it. This is called **dependency injection**, and it's simpler than the name:

The same works for:

- **chance**: pass in a `random.Random(seed)` object, and tests use a fixed seed;
- **input and output**: have a function **return** a result and let the caller print it; take text rather than calling `input()`;
- **files**: take the text or a file path, and let tests use a temporary folder (Day 5 shows pytest's way of making one).

A common shape is a **pure core** (functions that take values and return values, easy to test) inside a thin **shell** that reads input, calls the core and prints. You test the core thoroughly and the shell lightly.

## Testing properties

Example tests check particular inputs. A **property test** checks a rule that should hold for **every** input, over many generated inputs. For a sort function, some properties:

- the result is in order;
- the result has exactly the same items as the input (same counts);
- sorting a sorted list changes nothing.

The `, items` after each assert puts the input in the failure message, so you can see which case broke. Small values (`-5` to `5`) and short lists make repeats and edge cases likely.

Useful kinds of property:

- **round trip**: `decode(encode(x)) == x`, `json.loads(json.dumps(d)) == d`;
- **invariant**: something that never changes (the total of money moved between accounts; the length of a list after shuffling);
- **comparison with a simple version**: a fast, clever function must agree with a slow, obviously correct one;
- **bounds**: a dice roll is always 1 to 6; a percentage is always 0 to 100.

The library **Hypothesis** does this professionally: it generates inputs, and when a test fails it shrinks the input to the simplest one that still fails. It's worth learning once you're writing tests regularly.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at a test you wrote this week. Is its expected value worked out independently? Would it break if you tidied the function without changing what it does? Does it depend on the time, on chance, or on another test? Tomorrow: test-driven development, where the tests come first.